In [1]:
# import pandas as pd

# # Đọc các file pickle từ thư mục ../data/
# train_df = pd.read_pickle('../data/train_fe.pkl')
# val_df = pd.read_pickle('../data/val_fe.pkl')
# test_df = pd.read_pickle('../data/test_fe.pkl')

# # Kiểm tra dữ liệu (ví dụ: in ra số dòng và cột)
# print(f"Train shape: {train_df.shape}")
# print(f"Val shape: {val_df.shape}")
# print(f"Test shape: {test_df.shape}")


In [ ]:
# from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, SelectPercentile
# from xgboost import XGBRegressor, XGBClassifier
# import pandas as pd
# import numpy as np
# import json

# # ════════════════════════════════════════════════════════════════
# # 1. CHUẨN BỊ DỮ LIỆU
# # ════════════════════════════════════════════════════════════════
# DROP_COLS = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']

# y_train = train_df['target_log_revenue']
# X_train = train_df.drop(columns=DROP_COLS)

# y_val = val_df['target_log_revenue']
# X_val = val_df.drop(columns=DROP_COLS)

# print(f"Số lượng đặc trưng ban đầu: {len(X_train.columns)}")

# # ════════════════════════════════════════════════════════════════
# # LỚP 1: VARIANCE THRESHOLD
# # ════════════════════════════════════════════════════════════════
# var_selector = VarianceThreshold(threshold=(.995 * (1 - .995)))
# var_selector.fit(X_train)

# X_train_v = X_train.loc[:, var_selector.get_support()]
# X_val_v   = X_val.loc[:, var_selector.get_support()]
# print(f"1. Sau Variance Threshold: Còn {X_train_v.shape[1]} biến")

# # ════════════════════════════════════════════════════════════════
# # LỚP 2: CORRELATION FILTER
# # ════════════════════════════════════════════════════════════════
# corr_matrix = X_train_v.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

# X_train_c = X_train_v.drop(columns=to_drop_corr)
# X_val_c   = X_val_v.drop(columns=to_drop_corr)
# print(f"2. Sau Correlation Filter: Còn {X_train_c.shape[1]} biến")

# # ════════════════════════════════════════════════════════════════
# # LỚP 3: MUTUAL INFORMATION
# # ════════════════════════════════════════════════════════════════
# print("3. Đang thực hiện Mutual Information...")
# mi_percentile = 95 if X_train_c.shape[1] < 100 else 90
# mi_selector = SelectPercentile(mutual_info_regression, percentile=mi_percentile)

# # fillna(-1) chỉ dùng trong MI (không ảnh hưởng X_train_c gốc)
# mi_selector.fit(X_train_c.fillna(-1), y_train)  

# X_train_mi = X_train_c.loc[:, mi_selector.get_support()]
# X_val_mi   = X_val_c.loc[:, mi_selector.get_support()]
# print(f"3. Sau Mutual Information (Top {mi_percentile}%): Còn {X_train_mi.shape[1]} biến")

# # ════════════════════════════════════════════════════════════════
# # LỚP 4: 2-STAGE XGBOOST FEATURE IMPORTANCE (Union Approach)
# # ════════════════════════════════════════════════════════════════
# print("4. Đang chạy XGBoost Importance cho cả 2 tầng (Classification + Regression)...")

# # --- 4A. CLASSIFICATION IMPORTANCE (Ai sẽ mua?) ---
# y_train_buy = (y_train > 0).astype(int)
# y_val_buy   = (y_val > 0).astype(int)

# model_fs_clf = XGBClassifier(
#     n_estimators=500, max_depth=3, learning_rate=0.05, 
#     subsample=0.8, colsample_bytree=0.8,
#     random_state=42, n_jobs=-1,
#     early_stopping_rounds=30, eval_metric='logloss'
# )

# model_fs_clf.fit(
#     X_train_mi, y_train_buy,
#     eval_set=[(X_val_mi, y_val_buy)],
#     verbose=False
# )

# imp_clf = pd.DataFrame({
#     'feature': X_train_mi.columns,
#     'importance': model_fs_clf.feature_importances_
# }).sort_values(by='importance', ascending=False)

# # Cumulative 95% cho Classification
# cutoff_clf = (imp_clf['importance'].cumsum() <= 0.95).sum()
# keep_clf = imp_clf.iloc[:cutoff_clf + 1]['feature'].tolist()

# # --- 4B. REGRESSION IMPORTANCE (Mua bao nhiêu? - Chỉ dùng tập Buyers) ---
# mask_buyers_train = y_train > 0
# mask_buyers_val   = y_val > 0

# model_fs_reg = XGBRegressor(
#     n_estimators=500, max_depth=3, learning_rate=0.05, 
#     subsample=0.8, colsample_bytree=0.8,
#     random_state=42, n_jobs=-1,
#     early_stopping_rounds=30, eval_metric='rmse'
# )

# model_fs_reg.fit(
#     X_train_mi[mask_buyers_train], y_train[mask_buyers_train],
#     eval_set=[(X_val_mi[mask_buyers_val], y_val[mask_buyers_val])],
#     verbose=False
# )

# imp_reg = pd.DataFrame({
#     'feature': X_train_mi.columns,
#     'importance': model_fs_reg.feature_importances_
# }).sort_values(by='importance', ascending=False)

# # Cumulative 95% cho Regression
# cutoff_reg = (imp_reg['importance'].cumsum() <= 0.95).sum()
# keep_reg = imp_reg.iloc[:cutoff_reg + 1]['feature'].tolist()

# # --- 4C. GỘP (UNION) ĐẶC TRƯNG TỪ 2 TẦNG ---
# final_features = list(set(keep_clf) | set(keep_reg))

# print(f"   + Giữ lại từ Classifier: {len(keep_clf)} biến")
# print(f"   + Giữ lại từ Regressor:  {len(keep_reg)} biến")
# print(f"4. Sau Model Importance (Lấy Hợp - Union): Còn {len(final_features)} biến")

# # ════════════════════════════════════════════════════════════════
# # TỔNG KẾT VÀ LƯU TRỮ
# # ════════════════════════════════════════════════════════════════

# metadata_train_val = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']
# metadata_test      = ['fullVisitorId', 'Fold', 'Split_Type']

# train_df_final = train_df[metadata_train_val + final_features]
# val_df_final   = val_df[metadata_train_val + final_features]
# test_df_final  = test_df[metadata_test + final_features]

# # Lưu pickle
# train_df_final.to_pickle('../data/train_final.pkl')
# val_df_final.to_pickle('../data/val_final.pkl')
# test_df_final.to_pickle('../data/test_final.pkl')

# features_dict = {
#     'clf_features': keep_clf,
#     'reg_features': keep_reg,
#     'union_features': final_features
# }
# with open('../data/final_features.json', 'w') as f:
#     json.dump(features_dict, f, indent=2)


# print(f"\n=> HOÀN TẤT: Còn lại {len(final_features)} đặc trưng tinh túy nhất cho mô hình 2 tầng.")
# print("Dữ liệu cuối cùng đã được lưu thành công.")

# # In ra Top 10 của mỗi tầng để kiểm tra xem chúng có khác nhau nhiều không
# print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (CLASSIFICATION) ---")
# print(imp_clf.head(10))

# print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (REGRESSION) ---")
# print(imp_reg.head(10))


Stage 1: Classification

In [2]:
from xgboost import XGBClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    brier_score_loss,
    precision_recall_curve
)
import numpy as np
import pandas as pd
import json

# ════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ════════════════════════════════════════════════════════════════
train_df = pd.read_pickle('../data/train_final.pkl')
val_df   = pd.read_pickle('../data/val_final.pkl')

with open('../data/final_features.json') as f:
    features_dict = json.load(f)

clf_features = features_dict['clf_features']
X_train = train_df[clf_features]
X_val   = val_df[clf_features]

# ════════════════════════════════════════════════════════════════
# 2. BINARY TARGET
# ════════════════════════════════════════════════════════════════
y_train_buy = (train_df['target_revenue'] > 0).astype(int)
y_val_buy   = (val_df['target_revenue'] > 0).astype(int)

neg = (y_train_buy == 0).sum()
pos = (y_train_buy == 1).sum()
spw = np.cbrt(neg / pos)

print("=== Class Distribution ===")
print(y_train_buy.value_counts(normalize=True).round(6))
print(f"\nNegative: {neg:,} | Positive: {pos:,}")
print(f"SPW (cbrt): {spw:.2f}x")

# ════════════════════════════════════════════════════════════════
# 3. TEMPORAL OOF CALIBRATION
#
# Vì data chia theo time-based folds, KHÔNG dùng KFold shuffle.
# Thay vào đó dùng thứ tự Fold tự nhiên:
#   - Các Fold TRAIN sớm hơn = calibration train
#   - Fold TRAIN muộn nhất   = calibration val
# → Không có temporal leakage
# ════════════════════════════════════════════════════════════════
train_folds = sorted(train_df['Fold'].unique())
n_folds     = len(train_folds)

# Chia theo tỉ lệ 80/20 theo thứ tự thời gian
n_cal_train = int(n_folds * 0.8)

cal_train_folds = train_folds[:n_cal_train]
cal_val_folds   = train_folds[n_cal_train:]

cal_train_mask = train_df['Fold'].isin(cal_train_folds)
cal_val_mask   = train_df['Fold'].isin(cal_val_folds)

print(f"\n=== Temporal Calibration Split ===")
print(f"Train folds:       {train_folds}")
print(f"Cal train folds:   {cal_train_folds}")
print(f"Cal val folds:     {cal_val_folds}")
print(f"Cal train rows:    {cal_train_mask.sum():,}")
print(f"Cal val rows:      {cal_val_mask.sum():,}")

X_cal_train = X_train[cal_train_mask]
y_cal_train = y_train_buy[cal_train_mask]
X_cal_val   = X_train[cal_val_mask]
y_cal_val   = y_train_buy[cal_val_mask]

# ════════════════════════════════════════════════════════════════
# 4. ENSEMBLE + TEMPORAL CALIBRATION
# ════════════════════════════════════════════════════════════════
N_RUNS         = 5
p_cal_sum      = np.zeros(len(val_df))
pr_auc_raw_list = []
pr_auc_cal_list = []
brier_raw_list  = []
brier_cal_list  = []

print(f"\nTraining {N_RUNS} classifiers with temporal OOF calibration...")

for seed in range(N_RUNS):

    # ────────────────────────────────────────
    # BƯỚC 1: Train base model trên toàn train
    # ────────────────────────────────────────
    clf = XGBClassifier(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric='aucpr',
        early_stopping_rounds=30,
        random_state=seed,
        n_jobs=-1
    )
    clf.fit(
        X_train, y_train_buy,
        eval_set=[(X_val, y_val_buy)],
        verbose=False
    )

    # ────────────────────────────────────────
    # BƯỚC 2: Tạo OOF predictions theo thứ tự thời gian
    # Cal train folds → fit isotonic
    # Cal val folds   → predict để calibrate
    # Không có shuffle → không có temporal leakage
    # ────────────────────────────────────────
    p_cal_val_oof = clf.predict_proba(X_cal_val)[:, 1]

    # ────────────────────────────────────────
    # BƯỚC 3: Fit Isotonic trên cal_val_oof
    # (dùng predictions trên fold muộn hơn để calibrate)
    # ────────────────────────────────────────
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_cal_val_oof, y_cal_val)

    # ────────────────────────────────────────
    # BƯỚC 4: Apply lên val — không leakage
    # ────────────────────────────────────────
    p_raw = clf.predict_proba(X_val)[:, 1]
    p_cal = iso.predict(p_raw)

    p_cal_sum += p_cal

    # Metrics
    pr_raw = average_precision_score(y_val_buy, p_raw)
    pr_cal = average_precision_score(y_val_buy, p_cal)
    b_raw  = brier_score_loss(y_val_buy, p_raw)
    b_cal  = brier_score_loss(y_val_buy, p_cal)

    pr_auc_raw_list.append(pr_raw)
    pr_auc_cal_list.append(pr_cal)
    brier_raw_list.append(b_raw)
    brier_cal_list.append(b_cal)

    print(f"  seed={seed} | "
          f"PR raw={pr_raw:.4f} cal={pr_cal:.4f} | "
          f"Brier raw={b_raw:.6f} cal={b_cal:.6f} | "
          f"best_iter={clf.best_iteration}")

# ════════════════════════════════════════════════════════════════
# 5. ENSEMBLE AVERAGE
# ════════════════════════════════════════════════════════════════
p_avg = p_cal_sum / N_RUNS
val_df['p_buy'] = p_avg

ensemble_pr_cal    = average_precision_score(y_val_buy, p_avg)
ensemble_brier_cal = brier_score_loss(y_val_buy, p_avg)

print(f"\n=== Ensemble Results ===")
print(f"PR-AUC raw:  {np.mean(pr_auc_raw_list):.4f} ± {np.std(pr_auc_raw_list):.4f}")
print(f"PR-AUC cal:  {np.mean(pr_auc_cal_list):.4f} ± {np.std(pr_auc_cal_list):.4f}")
print(f"Brier raw:   {np.mean(brier_raw_list):.6f}")
print(f"Brier cal:   {np.mean(brier_cal_list):.6f}")
print(f"\nEnsemble PR-AUC (calibrated): {ensemble_pr_cal:.4f}")
print(f"Ensemble Brier (calibrated):  {ensemble_brier_cal:.6f}")

# ════════════════════════════════════════════════════════════════
# 6. CALIBRATION QUALITY CHECK
# ════════════════════════════════════════════════════════════════
print(f"\n=== Calibration Check ===")
print(f"Mean p_buy:      {p_avg.mean():.6f}")
print(f"True buyer rate: {y_val_buy.mean():.6f}")
print(f"Ratio:           {p_avg.mean()/y_val_buy.mean():.2f}x  (lý tưởng = 1.0x)")

# ════════════════════════════════════════════════════════════════
# 7. TÌM OPTIMAL THRESHOLD bằng F2
# Recall quan trọng hơn Precision (beta=2)
# Vì chi phí bỏ sót buyer >> chi phí nhầm non-buyer
# ════════════════════════════════════════════════════════════════
thresholds = np.arange(0.001, 0.5, 0.001)
f2_scores  = []

for t in thresholds:
    y_pred = (p_avg >= t).astype(int)
    f2 = fbeta_score(y_val_buy, y_pred, beta=2, zero_division=0)
    f2_scores.append(f2)

best_idx          = np.argmax(f2_scores)
optimal_threshold = thresholds[best_idx]
best_f2           = f2_scores[best_idx]

print(f"\n=== Optimal Threshold (Max F2, beta=2) ===")
print(f"Threshold: {optimal_threshold:.4f}")
print(f"F2 Score:  {best_f2:.4f}")
print(f"(F2 ưu tiên Recall gấp 2 lần Precision)")

# ════════════════════════════════════════════════════════════════
# 8. CLASSIFICATION REPORT
# ════════════════════════════════════════════════════════════════
y_pred = (p_avg >= optimal_threshold).astype(int)

print(f"\n=== Classification Report (threshold={optimal_threshold:.4f}) ===")
print(classification_report(y_val_buy, y_pred, digits=4))

cm = confusion_matrix(y_val_buy, y_pred)
tn, fp, fn, tp = cm.ravel()
print("=== Confusion Matrix ===")
print(cm)
print(f"\nTP: {tp:,} ({tp/(tp+fn)*100:.1f}% buyers found)    ← Recall")
print(f"FN: {fn:,} ({fn/(tp+fn)*100:.1f}% buyers missed)   ← muốn tối thiểu")
print(f"FP: {fp:,} ({fp/(fp+tn)*100:.2f}% non-buyers nhầm) ← chấp nhận được")
print(f"TN: {tn:,}")

print("\n✅ Saved: val_df['p_buy'] | optimal_threshold")

=== Class Distribution ===
target_revenue
0    0.999071
1    0.000929
Name: proportion, dtype: float64

Negative: 1,575,674 | Positive: 1,465
SPW (cbrt): 10.25x

=== Temporal Calibration Split ===
Train folds:       [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Cal train folds:   [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Cal val folds:     [np.int64(5)]
Cal train rows:    1,284,948
Cal val rows:      292,191

Training 5 classifiers with temporal OOF calibration...
  seed=0 | PR raw=0.0781 cal=0.0680 | Brier raw=0.001774 cal=0.000718 | best_iter=145
  seed=1 | PR raw=0.0764 cal=0.0677 | Brier raw=0.001690 cal=0.000718 | best_iter=58
  seed=2 | PR raw=0.0777 cal=0.0648 | Brier raw=0.001725 cal=0.000719 | best_iter=86
  seed=3 | PR raw=0.0738 cal=0.0644 | Brier raw=0.003823 cal=0.000716 | best_iter=17
  seed=4 | PR raw=0.0781 cal=0.0683 | Brier raw=0.001776 cal=0.000720 | best_iter=137

=== Ensemble Results ===
PR-AUC raw:  0.0768 ± 0.0017
PR-AUC cal:  0.066

Stage 2: Regression

In [3]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

reg_features = features_dict['reg_features']

# ════════════════════════════════════════════════════════════════
# 1. BUYERS ONLY — cho buyers regression
# ════════════════════════════════════════════════════════════════
train_buyers = train_df[train_df['target_revenue'] > 0].copy()
val_buyers   = val_df[val_df['target_revenue'] > 0].copy()

X_train_b = train_buyers[reg_features]
X_val_b   = val_buyers[reg_features]
y_train_b = train_buyers['target_log_revenue']
y_val_b   = val_buyers['target_log_revenue']

p99_log = y_train_b.quantile(0.99)

print(f"Train buyers: {len(train_buyers):,} ({len(train_buyers)/len(train_df)*100:.2f}%)")
print(f"Val buyers:   {len(val_buyers):,} ({len(val_buyers)/len(val_df)*100:.2f}%)")
print(f"Log revenue p99 (clip ceiling): {p99_log:.4f}")

# ════════════════════════════════════════════════════════════════
# 2. SAMPLE WEIGHTS — clip p99, normalize
# Focus high-value buyers nhưng không để 1 user dominate
# ════════════════════════════════════════════════════════════════
weights_raw = np.expm1(y_train_b.values)
p99_w       = np.percentile(weights_raw, 99)
weights     = np.clip(weights_raw, 0, p99_w)
weights     = weights / weights.mean()

print(f"\nSample weights: min={weights.min():.2f} | "
      f"max={weights.max():.2f} | mean={weights.mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 3. ENSEMBLE BUYERS REGRESSION
# Predict E[log_revenue | buyer]
# ════════════════════════════════════════════════════════════════
N_RUNS    = 5
e_sum_val = np.zeros(len(val_df))
e_sum_buy = np.zeros(len(val_buyers))
rmse_b_list = []

print(f"\nTraining {N_RUNS} buyer regressors...")

for seed in range(N_RUNS):
    reg_b = XGBRegressor(
        n_estimators=2000,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric='rmse',
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1
    )
    reg_b.fit(
        X_train_b, y_train_b,
        sample_weight=weights,
        eval_set=[(X_val_b, y_val_b)],
        verbose=False
    )

    pred_all = reg_b.predict(val_df[reg_features]).clip(0, p99_log)
    e_sum_val += pred_all
    e_sum_buy += pred_all[val_df['target_revenue'] > 0]

    rmse = np.sqrt(mean_squared_error(y_val_b,
                   pred_all[val_df['target_revenue'] > 0]))
    rmse_b_list.append(rmse)
    print(f"  seed={seed} | RMSE buyers={rmse:.4f} | best_iter={reg_b.best_iteration}")

E_log_buyers = (e_sum_val / N_RUNS)   # predict E[log_rev] cho toàn bộ val

# ════════════════════════════════════════════════════════════════
# 4. ENSEMBLE SINGLE REGRESSION (ALL USERS)
# Predict CLV trực tiếp, không cần tầng classification
# Dùng để so sánh với 2-stage
# ════════════════════════════════════════════════════════════════
e_sum_all = np.zeros(len(val_df))
rmse_all_list = []

print(f"\nTraining {N_RUNS} single regressors (ALL users)...")

for seed in range(N_RUNS):
    reg_all = XGBRegressor(
        n_estimators=2000,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric='rmse',
        early_stopping_rounds=50,
        random_state=seed,
        n_jobs=-1
    )
    reg_all.fit(
        train_df[reg_features],
        train_df['target_log_revenue'],
        eval_set=[(val_df[reg_features],
                   val_df['target_log_revenue'])],
        verbose=False
    )

    pred = reg_all.predict(val_df[reg_features]).clip(0)
    e_sum_all += pred

    rmse = np.sqrt(mean_squared_error(
        val_df['target_log_revenue'], pred))
    rmse_all_list.append(rmse)
    print(f"  seed={seed} | RMSE all={rmse:.4f} | best_iter={reg_all.best_iteration}")

E_log_all = (e_sum_all / N_RUNS)

# ════════════════════════════════════════════════════════════════
# 5. EVALUATE BUYERS REGRESSION
# ════════════════════════════════════════════════════════════════
baseline_b = np.sqrt(mean_squared_error(y_val_b, np.zeros(len(y_val_b))))
ensemble_b = np.sqrt(mean_squared_error(
    y_val_b, (e_sum_buy/N_RUNS).clip(0, p99_log)))

print(f"\n=== Stage 2: Buyers Regression ===")
print(f"RMSE per seed:  {[f'{x:.4f}' for x in rmse_b_list]}")
print(f"RMSE mean:      {np.mean(rmse_b_list):.4f} ± {np.std(rmse_b_list):.4f}")
print(f"Ensemble RMSE:  {ensemble_b:.4f}")
print(f"Baseline RMSE:  {baseline_b:.4f}")
print(f"Improvement:    {baseline_b - ensemble_b:.4f}")

print(f"\n=== Stage 2: Single Regression (ALL) ===")
print(f"RMSE per seed:  {[f'{x:.4f}' for x in rmse_all_list]}")
print(f"RMSE mean:      {np.mean(rmse_all_list):.4f} ± {np.std(rmse_all_list):.4f}")

# Lưu cả 2 để dùng trong combine
val_df['e_log_revenue']     = E_log_buyers   # buyers-only prediction
val_df['e_log_revenue_all'] = E_log_all      # single regression prediction

print("\n✅ Saved: val_df['e_log_revenue'] | val_df['e_log_revenue_all'] | p99_log")

Train buyers: 1,465 (0.09%)
Val buyers:   251 (0.07%)
Log revenue p99 (clip ceiling): 21.6405

Sample weights: min=0.00 | max=10.77 | mean=1.00

Training 5 buyer regressors...
  seed=0 | RMSE buyers=1.3840 | best_iter=1443
  seed=1 | RMSE buyers=1.4056 | best_iter=1061
  seed=2 | RMSE buyers=1.3994 | best_iter=1737
  seed=3 | RMSE buyers=1.4021 | best_iter=1629
  seed=4 | RMSE buyers=1.4059 | best_iter=1486

Training 5 single regressors (ALL users)...
  seed=0 | RMSE all=0.4825 | best_iter=43
  seed=1 | RMSE all=0.4825 | best_iter=61
  seed=2 | RMSE all=0.4821 | best_iter=60
  seed=3 | RMSE all=0.4826 | best_iter=52
  seed=4 | RMSE all=0.4826 | best_iter=43

=== Stage 2: Buyers Regression ===
RMSE per seed:  ['1.3840', '1.4056', '1.3994', '1.4021', '1.4059']
RMSE mean:      1.3994 ± 0.0081
Ensemble RMSE:  1.3946
Baseline RMSE:  18.0156
Improvement:    16.6210

=== Stage 2: Single Regression (ALL) ===
RMSE per seed:  ['0.4825', '0.4825', '0.4821', '0.4826', '0.4826']
RMSE mean:      0.4

Combine and evaluate

In [7]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
# ════════════════════════════════════════════════════════════════
# 1. LOAD VALUES
# ════════════════════════════════════════════════════════════════
P        = val_df['p_buy'].values
E_log    = val_df['e_log_revenue'].values   # đã clip p99
log_true = val_df['target_log_revenue'].values
y_true   = val_df['target_revenue'].values
buyers_mask     = y_true > 0
non_buyers_mask = ~buyers_mask
baseline_rmse = np.sqrt(mean_squared_error(log_true, np.zeros(len(log_true))))
# ════════════════════════════════════════════════════════════════
# 2. FINAL COMBINE (CHUẨN TOÁN HỌC)
# ════════════════════════════════════════════════════════════════
# Chuyển E_log về revenue thực, tính kỳ vọng, rồi log lại
rev_buy = np.expm1(E_log)
final_revenue_pred = P * rev_buy
best_pred = np.log1p(final_revenue_pred).clip(0)
# Đánh giá RMSE
rmse    = np.sqrt(mean_squared_error(log_true, best_pred))
b_rmse  = np.sqrt(mean_squared_error(log_true[buyers_mask],  best_pred[buyers_mask]))
nb_rmse = np.sqrt(mean_squared_error(log_true[~buyers_mask], best_pred[~buyers_mask]))
# ════════════════════════════════════════════════════════════════
# 3. BẢNG SO SÁNH
# ════════════════════════════════════════════════════════════════
print(f"\n{'Approach':<35} {'RMSE':>8} {'vs Baseline':>12} {'Buyers':>8} {'NonBuyers':>10}")
print("-" * 78)
print(f"{'Baseline (toàn 0)':<35} {baseline_rmse:>8.4f} {'—':>12} {'—':>8} {'—':>10}")
print(f"{'Model (P × expm1(E_log))':<35} {rmse:>8.4f} {baseline_rmse - rmse:>+12.4f} {b_rmse:>8.4f} {nb_rmse:>10.4f}")
# ════════════════════════════════════════════════════════════════
# 4. SAVE FINAL
# ════════════════════════════════════════════════════════════════
val_df['final_log_clv_pred'] = best_pred
val_df['final_revenue_pred'] = final_revenue_pred
# ════════════════════════════════════════════════════════════════
# 5. MARKETING METRICS
# ════════════════════════════════════════════════════════════════
top_k   = int(len(val_df) * 0.10)
top_idx = np.argsort(best_pred)[::-1][:top_k]
revenue_capture   = y_true[top_idx].sum() / (y_true.sum() + 1e-9)
buyer_capture     = buyers_mask[top_idx].mean()
overall_buyer_rate = buyers_mask.mean()
lift = buyer_capture / (overall_buyer_rate + 1e-9)

print(f"\n=== Marketing Metrics ===")
print(f"Revenue Capture@10%:  {revenue_capture:.4f}")
print(f"Buyer Rate@10%:       {buyer_capture:.4f}")
print(f"Overall Buyer Rate:   {overall_buyer_rate:.6f}")
print(f"Lift@10%:             {lift:.2f}x")

# ════════════════════════════════════════════════════════════════
# 5. TOP 10 USERS
# ════════════════════════════════════════════════════════════════
top10 = (
    val_df[['p_buy', 'e_log_revenue',
            'final_log_clv_pred', 'final_revenue_pred',
            'target_revenue', 'target_log_revenue']]
    .assign(is_buyer=(val_df['target_revenue'] > 0).astype(int))
    .sort_values('final_revenue_pred', ascending=False)
    .head(10)
)

pd.set_option('display.float_format', '{:,.4f}'.format)
print(f"\n=== Top 10 Predicted CLV Users ===")
print(top10.to_string())

n_buyers_top10   = top10['is_buyer'].sum()
revenue_cap_top10 = top10['target_revenue'].sum() / (y_true.sum() + 1e-9)

print(f"\n=== Top 10 Summary ===")
print(f"True buyers in top10:        {n_buyers_top10}/10")
print(f"Revenue captured by top10:   {revenue_cap_top10:.4%}")
print(f"Avg predicted revenue:       {top10['final_revenue_pred'].mean():,.0f}")
print(f"Avg true revenue:            {top10['target_revenue'].mean():,.0f}")

print("\n✅ Saved: val_df['final_log_clv_pred'] | val_df['final_revenue_pred']")



Approach                                RMSE  vs Baseline   Buyers  NonBuyers
------------------------------------------------------------------------------
Baseline (toàn 0)                     0.4915            —        —          —
Model (P × expm1(E_log))              8.0484      -7.5569   4.2662     8.0505

=== Marketing Metrics ===
Revenue Capture@10%:  0.9693
Buyer Rate@10%:       0.0069
Overall Buyer Rate:   0.000744
Lift@10%:             9.24x

=== Top 10 Predicted CLV Users ===
         p_buy  e_log_revenue  final_log_clv_pred  final_revenue_pred   target_revenue  target_log_revenue  is_buyer
1643138 0.9964        21.6405             21.6368  2,493,163,191.6200 330,690,000.0000             19.6167         1
1869041 0.8469        20.1873             20.0211    495,514,175.5022           0.0000              0.0000         0
1606177 0.3238        20.8742             19.7466    376,551,274.1643 461,940,000.0000             19.9509         1
1823882 0.2920        20.6525         